<a href="https://colab.research.google.com/github/kyore0382/gwagwayun/blob/main/docs/notebooks/Training_and_inference_using_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training and inference on your own data using Google Drive

In this notebook we'll install [`sleap-nn`](https://nn.sleap.ai/latest), import training data into Colab using [Google Drive](https://www.google.com/drive), and run training and inference.

## Install `sleap-nn``


In [1]:
import torch
print(torch.cuda.is_available())

True


## Import training data into Colab with Google Drive
We'll first prepare and export the training data from SLEAP, then upload it to Google Drive, and then mount Google Drive  into this Colab notebook.

### Create and export the training job package
A self-contained **training job package** contains a .slp file with labeled data and images which will be used for training, as well as .yaml training configuration file(s).

A training job package can be exported in the SLEAP GUI fron the "Run Training.." dialog under the "Predict" menu.

### Upload training job package to Google Drive
To be consistent with the examples in this notebook, name the SLEAP project `colab` and create a directory called `sleap` in the root of your Google Drive. Then upload the exported training job package `colab.slp.training_job.zip` into `sleap` directory.

If you place your training pckage somewhere else, or name it differently, adjust the paths/filenames/parameters below accordingly.

### Mount your Google Drive
Mounting your Google Drive will allow you to accessed the uploaded training job package in this notebook. When prompted to log into your Google account, give Colab access and the copy the authorization code into a field below (+ hit 'return').

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Let's set your current working directory to the directory with your training job package and unpack it there. Later on the output from training (i.e., the models) and from interence (i.e., predictions) will all be saved in this directory as well.

In [3]:
!pip install torch torchvision --quiet
!pip install sleap-nn --quiet
!pip install "sleap[nn]==1.6.2" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.6/434.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.1/731.1 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.4/848.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import sleap
print(sleap.__version__)

1.6.2


## Train a model

Let's train a model with the training profile (.yaml file) and the project data (.slp file) you have exported from SLEAP.


### Note on training profiles
Depending on the pipeline you chose in the training dialog, the config filename(s) will be:

- for a **bottom-up** pipeline approach: `bottomup.yaml` (this is the pipeline we assume here),

- for a **top-down** pipeline, you'll have a different profile for each of the models: `centroid.yaml` and `centered_instance.yaml`,

- for a **single animal** pipeline: `single_instance.yaml`.


### Note on training process
When you start training, you'll first see the training parameters and then the training and validation loss for each training epoch.

As soon as you're satisfied with the validation loss you see for an epoch during training, you're welcome to stop training by clicking the stop button. The version of the model with the lowest validation loss is saved during training, and that's what will be used for inference.

If you don't stop training, it will run for 200 epochs or until validation loss fails to improve for some number of epochs (controlled by the early_stopping fields in the training profile).

In [5]:
import sleap
print(sleap.__version__)

1.6.2


If instead of bottom-up you've chosen the top-down pipeline (with two training configs), you would need to invoke two separate training jobs in sequence:

`!sleap-nn train --config centroid.yaml "data_config.train_labels_path=[colab.pkg.slp]" trainer_config.ckpt_dir="." trainer_config.run_name="colab_demo.centroid"`

`!sleap-nn train --config centered_instance.yaml "data_config.train_labels_path=[colab.pkg.slp]" trainer_config.ckpt_dir="." trainer_config.run_name="colab_demo.centered_instance"`

## Run inference to predict instances

Once training finishes, you'll see a new directory (or two new directories for top-down training pipeline) containing all the model files SLEAP needs to use for inference.

Here we'll use the created model files to run inference in two modes:

- predicting instances in suggested frames from the exported .slp file

- predicting and tracking instances in uploaded video

You can also download the trained models for running inference from the SLEAP GUI on your computer (or anywhere else).

### Predicting instances in suggested frames
This mode of predicting instances is useful for accelerating the manual labeling work; it allows you to get early predictions on suggested frames and merge them back into the project for faster labeling.

Here we assume you've trained a bottom-up model and that the model files were written in directory named `colab_demo.bottomup`; later in this notebook we'll also show how to run inference with the pair of top-down models instead.

In [6]:
!sleap train "/content/drive/MyDrive/sleap/colab.pkg.slp"

INFO:numexpr.utils:NumExpr defaulting to 2 threads.
╭─────────────────────────────── sleap-nn train ───────────────────────────────╮
│ Usage                                                                        │
│                                                                              │
│                                                                              │
│  sleap-nn train <config.yaml> [overrides]                                    │
│  sleap-nn train --config <path/to/config.yaml> [overrides]                   │
│                                                                              │
│                                                                              │
│ Common Overrides                                                             │
│                                                                              │
│                                                                              │
│  Override                       Description            

Now, you can download the generated `colab.predicted_suggestions.slp` file and merge it into your labeling project (**File -> Merge into Project...** from the GUI) to get new predictions for your suggested frames.

### Predicting and tracking instances in uploaded video
Let's first upload the video we want to run inference on and name it `colab_demo.mp4`. (If your video is not named `colab_demo.mp4`, adjust the names below accordingly.)

For this demo we'll just get predictions for the first 200 frames (or you can adjust the --frames parameter below or remove it to run on the whole video).

In [10]:
import sleap_io as sio

labels = sio.load_slp("/content/drive/MyDrive/sleap/colab.pkg.slp")

print("노드 수:", len(labels.skeleton.nodes))
print("프레임 수:", len(labels))
print("스켈레톤:", labels.skeleton)

노드 수: 11
프레임 수: 38
스켈레톤: Skeleton(nodes=["head", "body", "chest", "R_S", "L_S", "R A", "L_A", "L_H", "R_L", "L_L", "R_H"], edges=[(0, 2), (2, 1), (2, 3), (2, 4), (3, 5), (4, 6), (1, 8), (1, 9), (6, 7), (5, 10)])


In [9]:
import sleap_io as sio
print(dir(sio))

['AnnotationType', 'Camera', 'CameraGroup', 'Edge', 'FrameGroup', 'Instance', 'InstanceContext', 'InstanceGroup', 'LabeledFrame', 'Labels', 'LabelsSet', 'MatchResult', 'Node', 'PredictedInstance', 'ROI', 'RecordingSession', 'RenderContext', 'SegmentationMask', 'Skeleton', 'SuggestionFrame', 'Symmetry', 'Track', 'Transform', 'Video', 'VideoBackend', 'VideoWriter', 'codecs', 'get_available_image_backends', 'get_available_video_backends', 'get_default_image_plugin', 'get_default_video_plugin', 'get_installation_instructions', 'get_palette', 'io', 'load_alphatracker', 'load_analysis_h5', 'load_coco', 'load_csv', 'load_dlc', 'load_file', 'load_jabs', 'load_labels_set', 'load_labelstudio', 'load_leap', 'load_nwb', 'load_skeleton', 'load_slp', 'load_ultralytics', 'load_video', 'model', 'render_image', 'render_video', 'save_analysis_h5', 'save_coco', 'save_csv', 'save_file', 'save_jabs', 'save_labelstudio', 'save_nwb', 'save_skeleton', 'save_slp', 'save_ultralytics', 'save_video', 'set_default

In [11]:
import sleap_nn
train_funcs = [x for x in dir(sleap_nn) if 'train' in x.lower() or 'config' in x.lower()]
print(train_funcs)

[]


In [12]:
import sleap_nn
print(dir(sleap_nn))

['RANK', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_safe_print', '_should_log', 'evaluation', 'load_metrics', 'logger', 'os', 'sys']


In [13]:
import sleap_nn.training
print(dir(sleap_nn.training))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [14]:
import pkgutil
import sleap_nn

for importer, modname, ispkg in pkgutil.walk_packages(
    path=sleap_nn.__path__,
    prefix=sleap_nn.__name__+'.',
    onerror=lambda x: None
):
    print(modname)

sleap_nn.architectures
sleap_nn.architectures.common
sleap_nn.architectures.convnext
sleap_nn.architectures.encoder_decoder
sleap_nn.architectures.heads
sleap_nn.architectures.model
sleap_nn.architectures.swint
sleap_nn.architectures.unet
sleap_nn.architectures.utils
sleap_nn.cli
sleap_nn.config
sleap_nn.config.data_config
sleap_nn.config.get_config
sleap_nn.config.model_config
sleap_nn.config.trainer_config
sleap_nn.config.training_job_config
sleap_nn.config.utils
sleap_nn.config_generator
sleap_nn.config_generator.analyzer
sleap_nn.config_generator.generator
sleap_nn.config_generator.memory
sleap_nn.config_generator.recommender
sleap_nn.config_generator.tui
sleap_nn.config_generator.tui.app
sleap_nn.config_generator.tui.screens
sleap_nn.config_generator.tui.screens.configure_screen
sleap_nn.config_generator.tui.screens.data_screen
sleap_nn.config_generator.tui.screens.export_screen
sleap_nn.config_generator.tui.screens.load_screen
sleap_nn.config_generator.tui.screens.model_screen
sl

In [15]:
from sleap_nn.config_generator import generator
print(dir(generator))

['BackboneType', 'ConfigGenerator', 'ConfigRecommendation', 'DatasetStats', 'DictConfig', 'MemoryEstimate', 'OmegaConf', 'Optional', 'Path', 'PipelineType', 'TYPE_CHECKING', 'Tuple', 'Union', 'ViewType', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'analyze_slp', 'estimate_memory', 'recommend_config', 'recommend_pipeline']


In [16]:
import sleap_nn.train as trainer
print(dir(trainer))

['Any', 'Dict', 'DictConfig', 'List', 'ModelTrainer', 'OmegaConf', 'Optional', 'Path', 'TrainingJobConfig', 'Tuple', 'Union', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'datetime', 'get_data_config', 'get_model_config', 'get_startup_info_string', 'get_trainer_config', 'logger', 'predict', 'run_evaluation', 'run_training', 'sio', 'time', 'train']


In [17]:
from sleap_nn.config_generator.generator import analyze_slp, recommend_config

# slp 파일 분석
stats = analyze_slp("/content/drive/MyDrive/sleap/colab.pkg.slp")
print(stats)

# 최적 config 추천
config = recommend_config(stats)
print(config)

Dataset: colab.pkg.slp
  Labeled frames: 38
  Videos: 1
  Image size: 480x854 (RGB)
  Max instances/frame: 1
  Avg instances/frame: 0.8
  Max bbox size: 700.4px
  Avg bbox size: 603.8px
  Animal size: ~70.7% of frame
  Overlap frequency: 0.0%
  Skeleton: 11 nodes, 10 edges
  Tracks: none
ConfigRecommendation(pipeline=PipelineRecommendation(recommended='single_instance', reason='Only one animal detected per frame', alternatives=['centered_instance'], warnings=[], requires_second_model=False, second_model_type=None), backbone='unet_large_rf', backbone_reason='Large animals/images need larger receptive field (max_stride=32)', sigma=7.5, sigma_reason='Larger sigma for large animals (easier to learn)', input_scale=1.0, scale_reason='Image size suitable for full resolution', batch_size=8, batch_reason='Moderate image size allows larger batch', rotation_range=(-15.0, 15.0), rotation_reason='Default conservative rotation (specify view_type for better defaults)', crop_size=None, anchor_part=Non

In [19]:
import inspect
from sleap_nn.config_generator.generator import ConfigGenerator

print(inspect.signature(ConfigGenerator.__init__))

(self, slp_path: str)


In [21]:
print([x for x in dir(gen) if not x.startswith('_')])

['anchor_part', 'augmentation', 'auto', 'backbone', 'batch_size', 'build', 'build_centered_instance', 'crop_size', 'early_stopping', 'from_labels', 'from_slp', 'input_scale', 'is_topdown', 'learning_rate', 'max_epochs', 'memory_estimate', 'output_stride', 'pipeline', 'recommend', 'rotation', 'save', 'scale_augmentation', 'sigma', 'slp_path', 'stats', 'summary', 'to_yaml', 'validation_fraction']


In [23]:
gen.auto()
gen.build()
yaml_config = gen.to_yaml()
print(yaml_config)

data_config:
  train_labels_path:
  - /content/drive/MyDrive/sleap/colab.pkg.slp
  val_labels_path: []
  validation_fraction: 0.1
  user_instances_only: true
  data_pipeline_fw: torch_dataset
  preprocessing:
    ensure_rgb: true
    ensure_grayscale: false
    scale: 1.0
    crop_size: null
  use_augmentations_train: true
  augmentation_config:
    geometric:
      rotation_min: -15.0
      rotation_max: 15.0
      scale_min: 0.9
      scale_max: 1.1
      translate_width: 0.0
      translate_height: 0.0
      affine_p: 1.0
    intensity:
      brightness_limit: 0.0
      brightness_p: 0.0
      contrast_limit: 0.0
      contrast_p: 0.0
model_config:
  init_weights: default
  backbone_config:
    unet:
      in_channels: 3
      filters: 24
      filters_rate: 1.5
      max_stride: 64
      output_stride: 1
  head_configs:
    single_instance:
      confmaps:
        sigma: 7.5
        output_stride: 1
    centroid: null
    centered_instance: null
    bottomup: null
    multi_class_b

In [24]:
gen.save("/content/drive/MyDrive/sleap/config.yaml")
print("config 저장 완료!")

config 저장 완료!


In [27]:
import inspect
from sleap_nn.config.data_config import IntensityConfig
print(inspect.signature(IntensityConfig.__init__))

(self, uniform_noise_min: float = 0.0, uniform_noise_max: float = 0.04, uniform_noise_p: float = 0.0, gaussian_noise_mean: float = 0.0, gaussian_noise_std: float = 0.02, gaussian_noise_p: float = 0.0, contrast_min: float = 0.9, contrast_max: float = 1.1, contrast_p: float = 0.0, brightness_min: float = 0.9, brightness_max: float = 1.1, brightness_p: float = 0.0) -> None


In [28]:
from omegaconf import OmegaConf

config = OmegaConf.load("/content/drive/MyDrive/sleap/config.yaml")

# 잘못된 intensity 필드 수정
config.data_config.augmentation_config.intensity = {
    "uniform_noise_min": 0.0,
    "uniform_noise_max": 0.04,
    "uniform_noise_p": 0.0,
    "gaussian_noise_mean": 0.0,
    "gaussian_noise_std": 0.02,
    "gaussian_noise_p": 0.0,
    "contrast_min": 0.9,
    "contrast_max": 1.1,
    "contrast_p": 0.0,
    "brightness_min": 0.9,
    "brightness_max": 1.1,
    "brightness_p": 0.0
}

OmegaConf.save(config, "/content/drive/MyDrive/sleap/config.yaml")
print("config 수정 완료!")

config 수정 완료!


In [32]:
import inspect
from sleap_nn.config.trainer_config import TrainerConfig
print(inspect.signature(TrainerConfig.__init__))

(self, train_data_loader: sleap_nn.config.trainer_config.TrainDataLoaderConfig = NOTHING, val_data_loader: sleap_nn.config.trainer_config.ValDataLoaderConfig = NOTHING, model_ckpt: sleap_nn.config.trainer_config.ModelCkptConfig = NOTHING, trainer_devices: Optional[Any] = None, trainer_device_indices: Optional[List[int]] = None, trainer_accelerator: str = 'auto', profiler: Optional[str] = None, trainer_strategy: str = 'auto', enable_progress_bar: bool = True, min_train_steps_per_epoch: int = 200, train_steps_per_epoch: Optional[int] = None, visualize_preds_during_training: bool = False, keep_viz: bool = False, max_epochs: int = 100, seed: Optional[int] = None, use_wandb: bool = False, save_ckpt: bool = False, ckpt_dir: Optional[str] = '.', run_name: Optional[str] = None, resume_ckpt_path: Optional[str] = None, wandb: sleap_nn.config.trainer_config.WandBConfig = NOTHING, optimizer_name: str = 'Adam', optimizer: sleap_nn.config.trainer_config.OptimizerConfig = NOTHING, lr_scheduler: Optio

In [35]:
import inspect
from sleap_nn.config.trainer_config import ReduceLROnPlateauConfig, ModelCkptConfig
print(inspect.signature(ReduceLROnPlateauConfig.__init__))
print(inspect.signature(ModelCkptConfig.__init__))

(self, threshold: float = 1e-06, threshold_mode: str = 'abs', cooldown: int = 3, patience: int = 5, factor: float = 0.5, min_lr: Any = 1e-08) -> None
(self, save_top_k: int = 1, save_last: Optional[bool] = None) -> None


In [34]:
import inspect
from sleap_nn.config.trainer_config import LRSchedulerConfig
print(inspect.signature(LRSchedulerConfig.__init__))

(self, step_lr: Optional[sleap_nn.config.trainer_config.StepLRConfig] = None, reduce_lr_on_plateau: Optional[sleap_nn.config.trainer_config.ReduceLROnPlateauConfig] = NOTHING, cosine_annealing_warmup: Optional[sleap_nn.config.trainer_config.CosineAnnealingWarmupConfig] = None, linear_warmup_linear_decay: Optional[sleap_nn.config.trainer_config.LinearWarmupLinearDecayConfig] = None) -> None


In [38]:
from omegaconf import OmegaConf
from sleap_nn.train import run_training

config = OmegaConf.load("/content/drive/MyDrive/sleap/config.yaml")
OmegaConf.set_struct(config, False)

config.trainer_config.max_epochs = 50          # 200 → 50
config.trainer_config.min_train_steps_per_epoch = 50  # 200 → 50

OmegaConf.save(config, "/content/drive/MyDrive/sleap/config.yaml")
run_training(config)

2026-05-24 09:39:43 | Started training at: 2026-05-24 09:39:43.999175
2026-05-24 09:39:43 | sleap-nn 0.1.3 | Python 3.12.13 | PyTorch 2.10.0+cu128 | CUDA 12.8 | 1 GPU(s)
2026-05-24 09:39:44 | Creating train-val split...
2026-05-24 09:39:44 | # Train Labeled frames: 27
2026-05-24 09:39:44 | # Val Labeled frames: 3
2026-05-24 09:39:44 | Setting up config...
2026-05-24 09:39:44 | Setting up for training...
2026-05-24 09:39:44 | Setting up model ckpt dir: `260524_093944.single_instance.n=30`...
2026-05-24 09:39:44 | Setting up Trainer...
2026-05-24 09:39:44 | Setting up callbacks and loggers...
2026-05-24 09:39:44 | Trainer devices: auto


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


2026-05-24 09:39:45 | Training on 1 device(s)
2026-05-24 09:39:45 | Training on cuda:0 accelerator
2026-05-24 09:39:45 | Setting up lightning module for single_instance model...
2026-05-24 09:39:45 | Backbone model: UNet(
  (encoders): ModuleList(
    (0): Encoder(
      (encoder_stack): ModuleList(
        (0): SimpleConvBlock(
          (blocks): Sequential(
            (stack0_enc0_conv0): Conv2d(3, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
            (stack0_enc0_act0_relu): ReLU()
            (stack0_enc0_conv1): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
            (stack0_enc0_act1_relu): ReLU()
          )
        )
        (1): SimpleConvBlock(
          (blocks): Sequential(
            (stack0_enc1_pool): MaxPool2dWithSamePadding(kernel_size=2, stride=2, padding=same, dilation=1, ceil_mode=False)
            (stack0_enc1_conv0): Conv2d(24, 36, kernel_size=(3, 3), stride=(1, 1), padding=same)
            (stack0_enc1_act0_relu): ReLU()
      

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/260524_093944.single_instance.n=30 exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


2026-05-24 09:39:45 | Input image shape: torch.Size([1, 3, 896, 512])
2026-05-24 09:39:45 | Finished trainer set up. [0.9s]
2026-05-24 09:39:45 | Starting training loop...


...

2026-05-24 11:41:34 | Finished training loop. [121.8 min]
2026-05-24 11:41:34 | Finished training at: 2026-05-24 11:41:34.355420
2026-05-24 11:41:34 | Total training time: 7310.356261730194 secs
2026-05-24 11:41:34 | Training Config: data_config:
  train_labels_path:
  - /content/drive/MyDrive/sleap/colab.pkg.slp
  val_labels_path: []
  validation_fraction: 0.1
  use_same_data_for_val: false
  test_file_path: null
  provider: LabelsReader
  user_instances_only: true
  data_pipeline_fw: torch_dataset
  cache_img_path: null
  use_existing_imgs: false
  delete_cache_imgs_after_training: true
  parallel_caching: true
  cache_workers: 0
  preprocessing:
    ensure_rgb: true
    ensure_grayscale: false
    max_height: 854
    max_width: 480
    scale: 1.0
    crop_size: null
    min_crop_size: 100
    crop_padding: null
  use_augmentations_train: true
  augmentation_config:
    intensity:
      uniform_noise_min: 0.0
      uniform_noise_max: 0.04
      uniform_noise_p: 0.0
      gaussian_noi

Output()

2026-05-24 11:41:38 | Finished inference at: 2026-05-24 11:41:38.194067
2026-05-24 11:41:38 | Total runtime: 3.819035053253174 secs
2026-05-24 11:41:38 | Predictions output path: 260524_093944.single_instance.n=30/labels_pr.train.0.slp
2026-05-24 11:41:38 | Saved file at: 2026-05-24 11:41:38.371370
2026-05-24 11:41:38 | Skipping eval on `train.0` dataset as there are no labeled frames...
2026-05-24 11:41:38 | Started inference at: 2026-05-24 11:41:38.372855
2026-05-24 11:41:38 | sleap-nn 0.1.3 | Python 3.12.13 | PyTorch 2.10.0+cu128 | CUDA 12.8 | 1 GPU(s)
2026-05-24 11:41:38 | Using device: cuda:0


Output()

2026-05-24 11:41:38 | Finished inference at: 2026-05-24 11:41:38.924554
2026-05-24 11:41:38 | Total runtime: 0.5517089366912842 secs
2026-05-24 11:41:38 | Predictions output path: 260524_093944.single_instance.n=30/labels_pr.val.0.slp
2026-05-24 11:41:38 | Saved file at: 2026-05-24 11:41:38.991859
2026-05-24 11:41:38 | Skipping eval on `val.0` dataset as there are no labeled frames...


When inference is finished, it will save the predictions in a file which can be opened in the GUI as a SLEAP project file. The file will be in the same directory as the video and the filename will be `{video filename}.predictions.slp`.

You can copy this file from your Google Drive to a local drive and open it in the SLEAP GUI app (or open it directly if you have your Google Drive mounted on your local machine). If the video is in the same directory as the predictions file, SLEAP will automatically find it; otherwise, you'll be prompted to locate the video (since the path to the video on your local machine will be different than the path to the video on Colab).

### Inference with top-down models

If you trained the pair of models needed for top-down inference, you can call `sleap-track` with `-m path/to/model` for each model, like so:

In [ ]:
!sleap-nn track -i colab_demo.mp4 \
    --frames 0-200 \
    --tracking \
    -m colab_demo.centroid \
    -m colab_demo.centered_instance